# Day 9 · 可视化进阶:Pandas 直接画(W2 第 2 天)

目标:从 DataFrame 直接 `.plot()`,groupby 之后马上成图,会调样式、画双轴。

今日节奏:40min 学习 + 15min 动手 + 5min 自检。

> 打开方式:JupyterLab 文件树里进 `ai-learning/练习/` 双击本文件,逐格 Shift+Enter。


In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| matplotlib", plt.matplotlib.__version__)
print("环境 OK!开始今天的练习 →")

rng = np.random.default_rng(5)
dates = pd.date_range("2026-01-01", periods=60, freq="D")
sales = pd.DataFrame({
    "date": dates,
    "region": rng.choice(["North", "South", "East", "West"], 60),
    "amount": np.round(rng.normal(500, 120, 60), 0),
})
sales["amount"] = sales["amount"] + sales["date"].dt.dayofweek * 20   # 周末加量
print(sales.head())


## 任务 1:df.plot(kind=...) 一行成图

- Series 或 DataFrame 直接 `.plot()`,默认折线
- `kind="bar" / "barh" / "hist" / "scatter"` 切换图型
- 想看"每天"这种维度,先 `groupby` 再画


In [ ]:
daily = sales.groupby("date")["amount"].sum()
ax = daily.plot(figsize=(7, 3), linewidth=1.5)
ax.set_ylabel("Amount")
ax.set_title("Daily sales")
ax.grid(True)
plt.show()

plt.figure(figsize=(6, 3))
region_total = sales.groupby("region")["amount"].sum()
region_total.plot(kind="barh", color="steelblue")
plt.xlabel("Amount")
plt.title("Sales by region")
plt.show()


## 任务 2:常用样式参数

`plot` 一步到位:`figsize` 尺寸、`color` 颜色、`rot` 刻度旋转、`grid` 网格、`style` 点线组合(如 `".-"`)。


In [ ]:
daily.plot(figsize=(7, 3), color="darkgreen", rot=45, grid=True, style=".-")
plt.title("Daily sales (styled)")
plt.show()


## 任务 3:双轴图(twinx)

一张图同时画两个量纲不同的指标:

- `ax1` 画金额(折线)
- `ax2 = ax1.twinx()` 共享 x 轴、右轴是自己的 → 画订单数(柱状)
- 两个轴分别 `legend`,别挤成一团


In [ ]:
daily_amount = sales.groupby("date")["amount"].sum()
daily_orders = sales.groupby("date")["amount"].count()

fig, ax1 = plt.subplots(figsize=(7, 3))
ax1.plot(daily_amount, color="steelblue", label="amount")
ax1.set_ylabel("Amount")
ax2 = ax1.twinx()
ax2.bar(daily_orders.index, daily_orders.values, alpha=0.25, color="orange", label="orders")
ax2.set_ylabel("Orders")
ax1.legend(loc="upper left")
ax2.legend(loc="upper right")
ax1.set_title("Amount vs orders per day")
plt.show()


## 任务 4:小挑战 🔥(两张图讲一个故事)

1. 各地区销售额占比 → 饼图 `kind="pie"`(带百分比 autopct)
2. 每日销售额 vs 7 日移动平均 → 趋势对比折线

做完口头总结一句:这组数据"在讲什么故事"?


In [ ]:
plt.figure(figsize=(8, 3.5))

plt.subplot(1, 2, 1)
region_total.plot(kind="pie", autopct="%1.1f%%", startangle=90, counterclock=False)
plt.ylabel("")
plt.title("Region share")

plt.subplot(1, 2, 2)
smooth = daily.rolling(7).mean()
plt.plot(daily, alpha=0.5, label="daily")
plt.plot(smooth, label="7-day avg", linewidth=2)
plt.legend()
plt.title("Trend")

plt.tight_layout()
plt.show()


## 自检清单(5 问,答不上就回看今天的格子)

1. 从 DataFrame 直接画图用什么? → `df.plot(kind=...)`
2. 先 groupby 再画是为了什么? → 把细粒度数据汇总成想看的维度(如每天)
3. barh 是什么? → 横向柱状图,类别名长时更好读
4. 双轴图怎么开? → `ax2 = ax1.twinx()`,两轴各画各的
5. 饼图适合什么? → 占比/组成,类别别超过五六个

## 📝 收盘动作

```powershell
cd D:\01_Study\ai-learning
git add -A; git commit -m "day9: pandas plotting"; git push
```

然后跟助手说"生成日志"。
